<a href="https://colab.research.google.com/github/Linda-Masia/MIT-805---Vincent-Mabuza-Linda-Masia/blob/main/Build_Processing_Set.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!git clone https://github.com/Linda-Masia/MIT-805---Vincent-Mabuza-Linda-Masia.git
%cd MIT-805---Vincent-Mabuza-Linda-Masia

Cloning into 'MIT-805---Vincent-Mabuza-Linda-Masia'...
remote: Enumerating objects: 81, done.
remote: Counting objects: 100% (81/81), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 81 (delta 23), reused 50 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (81/81), 2.95 MiB | 12.48 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/MIT-805---Vincent-Mabuza-Linda-Masia/MIT-805---Vincent-Mabuza-Linda-Masia


In [13]:
!cat src/download_data.py

from pathlib import Path
import requests


# ============================================================
# SETTINGS
# ============================================================

# Change this number to whatever total size you want.
# This is the COMBINED limit across all 4 datasets.
TARGET_GB = 12

TARGET_BYTES = TARGET_GB * 1_000_000_000

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

DATA_DIR = Path("data/raw")

DATASETS = {
    "yellow": "yellow_tripdata",
    "green": "green_tripdata",
    "fhv": "fhv_tripdata",
    "fhvhv": "fhvhv_tripdata",
}


# ============================================================
# CREATE FOLDERS
# ============================================================

for dataset in DATASETS:
    (DATA_DIR / dataset).mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# GET TOTAL SIZE OF ALL 4 FOLDERS
# ============================================================

def get_tot

In [14]:
!pip install -r requirements.txt --quiet

In [15]:
!python src/download_data.py


NYC TLC DATA DOWNLOADER
Target: 12 GB COMBINED
Current: 0.00 GB

===== 2026-01 =====

[YELLOW] yellow_tripdata_2026-01.parquet
    Downloading...
    Downloaded: yellow_tripdata_2026-01.parquet
    Combined total: 0.06 GB

[GREEN] green_tripdata_2026-01.parquet
    Downloading...
    Downloaded: green_tripdata_2026-01.parquet
    Combined total: 0.07 GB

[FHV] fhv_tripdata_2026-01.parquet
    Downloading...
    Downloaded: fhv_tripdata_2026-01.parquet
    Combined total: 0.09 GB

[FHVHV] fhvhv_tripdata_2026-01.parquet
    Downloading...
    Downloaded: fhvhv_tripdata_2026-01.parquet
    Combined total: 0.59 GB

===== 2025-12 =====

[YELLOW] yellow_tripdata_2025-12.parquet
    Downloading...
    Downloaded: yellow_tripdata_2025-12.parquet
    Combined total: 0.67 GB

[GREEN] green_tripdata_2025-12.parquet
    Downloading...
    Downloaded: green_tripdata_2025-12.parquet
    Combined total: 0.67 GB

[FHV] fhv_tripdata_2025-12.parquet
    Downloading...
    Downloaded: fhv_tripdata_2025-

In [16]:
!ls src/

check_size.py  download_data.py


In [17]:
!ls -R src/ notebooks/

notebooks/:
EDA.ipynb

src/:
check_size.py  download_data.py


In [18]:
!pip install nbconvert --quiet
!jupyter nbconvert --to script notebooks/EDA.ipynb --output EDA_extracted
!cat notebooks/EDA_extracted.py

[NbConvertApp] Converting notebook notebooks/EDA.ipynb to script
[NbConvertApp] Writing 12606 bytes to notebooks/EDA_extracted.py
#!/usr/bin/env python
# coding: utf-8

# ## 1. Import Required Libraries

# In[1]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import pyarrow.parquet as pq

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries loaded successfully")


# ## 2. Dataset Paths and Sampling Settings

# In[2]:


DATASETS = {
    "FHV": Path("../data/raw/fhv"),
    "FHVHV": Path("../data/raw/fhvhv"),
    "Green": Path("../data/raw/green"),
    "Yellow": Path("../data/raw/yellow"),
}

SAMPLE_ROWS_PER_DATASET = 250_000
RANDOM_STATE = 42

for name, path in DATASETS.items():
    files = sorted(path.glob("*.parquet"))
    print(f"{name:8} | {len(files):4} files | {path}")


# ## 3. Representative Sampling

# In[3]:


SAMPLE_C

In [23]:
"""
build_processing_set.py

The processing dataset (3.52 GB) was constructed by randomly sampling row groups (seed=42) from each of the four raw TLC datasets (Yellow, Green, FHV, FHV-HV), using the same representative sampling methodology applied during Part 1 exploratory analysis, to ensure methodological consistency between the two parts of the project
"""

from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import gc
import os

RAW_DIR = Path("data/raw")
PROCESSING_DIR = Path("data/processing")
PROCESSING_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "yellow": RAW_DIR / "yellow",
    "green": RAW_DIR / "green",
    "fhv": RAW_DIR / "fhv",
    "fhvhv": RAW_DIR / "fhvhv",
}

RANDOM_STATE = 42

# Target size PER DATASET (GB). Increased to 1.5GB so yellow and fhvhv can
# compensate for green and fhv, easily bringing the total above 3.0GB.
TARGET_GB_PER_DATASET = 1.5


def downcast_dtypes(df):
    """Reduce memory footprint by downcasting float columns only to avoid int overflow issues."""
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")
    # Skip aggressive integer downcasting to prevent schema casting failures with values exceeding int8/int16 ranges
    return df


def get_dir_size_gb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 ** 3)


def sample_dataset_to_target_size(name, folder, target_gb, random_state=42):
    """
    Randomly samples row groups from a dataset's raw parquet files (same
    approach as the Part 1 EDA notebook), writing out incrementally and
    stopping once the target size on disk is reached. Never holds more
    than a few row groups in memory at once. Aligns schemas to prevent
    crashes from evolving/missing columns.
    """
    rng = np.random.default_rng(random_state)
    files = sorted(Path(folder).glob("*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parquet files found in {folder}")

    # Build a list of (file, row_group_index) pairs and shuffle - same
    # row-group-level randomization as the original notebook.
    row_groups = []
    for file in files:
        pf = pq.ParquetFile(file)
        for rg in range(pf.num_row_groups):
            row_groups.append((file, rg))

    order = rng.permutation(len(row_groups))

    out_path = PROCESSING_DIR / f"{name}_processing.parquet"
    target_bytes = target_gb * (1024 ** 3)

    writer = None
    total_rows = 0
    writer_schema = None

    for idx in order:
        file, rg = row_groups[idx]
        pf = pq.ParquetFile(file)
        table = pf.read_row_group(rg)

        df_chunk = table.to_pandas()
        df_chunk = downcast_dtypes(df_chunk)

        # Aligns chunk schema with the expected schema for the ParquetWriter
        if writer_schema is not None:
            # Add missing columns
            for field in writer_schema:
                if field.name not in df_chunk.columns:
                    df_chunk[field.name] = np.nan
            # Reorder and cast types to match writer_schema
            col_order = [field.name for field in writer_schema]
            df_chunk = df_chunk[col_order]

        chunk_table = pa.Table.from_pandas(df_chunk, preserve_index=False)

        if writer is None:
            # Define the initial schema
            schema_fields = []
            for field in chunk_table.schema:
                # Avoid NullType which causes issues when other chunks have string data
                if pa.types.is_null(field.type):
                    schema_fields.append(pa.field(field.name, pa.string(), nullable=True))
                else:
                    schema_fields.append(field)
            writer_schema = pa.schema(schema_fields)
            chunk_table = chunk_table.cast(writer_schema)
            writer = pq.ParquetWriter(out_path, writer_schema)
        else:
            # Ensure exact schema match before appending
            chunk_table = chunk_table.cast(writer_schema)

        writer.write_table(chunk_table)
        total_rows += len(df_chunk)

        del df_chunk, table, chunk_table
        gc.collect()

        current_size = out_path.stat().st_size / (1024 ** 3)
        if current_size >= target_gb:
            break

    if writer is not None:
        writer.close()

    final_size_gb = out_path.stat().st_size / (1024 ** 3) if out_path.exists() else 0
    print(f"  {name}: {total_rows:,} rows written, {final_size_gb:.2f} GB -> {out_path}")
    return final_size_gb


if __name__ == "__main__":
    print("Building Part 2 processing dataset from raw TLC data...")
    print(f"Target per dataset: {TARGET_GB_PER_DATASET} GB "
          f"(combined target: {TARGET_GB_PER_DATASET * len(DATASETS)} GB)\n")

    results = {}
    for name, folder in DATASETS.items():
        print(f"Sampling {name}...")
        size_gb = sample_dataset_to_target_size(
            name, folder, target_gb=TARGET_GB_PER_DATASET, random_state=RANDOM_STATE
        )
        results[name] = size_gb

    total_size = get_dir_size_gb(PROCESSING_DIR)
    print("\n" + "=" * 50)
    print("PROCESSING SET SUMMARY")
    print("=" * 50)
    for name, size in results.items():
        print(f"  {name}: {size:.2f} GB")
    print(f"\nTotal processing set size: {total_size:.2f} GB")

    if total_size < 3.0:
        print("WARNING: total is below the 3GB minimum - increase "
              "TARGET_GB_PER_DATASET and re-run.")
    else:
        print("Target met (>= 3GB).")

    print("\nSuggested report sentence:")
    print(f'"The processing dataset ({total_size:.2f} GB) was constructed by '
          f'randomly sampling row groups (seed=42) from each of the four raw '
          f'TLC datasets (Yellow, Green, FHV, FHV-HV), using the same '
          f'representative sampling methodology applied during Part 1 '
          f'exploratory analysis, to ensure methodological consistency '
          f'between the two parts of the project."')


Building Part 2 processing dataset from raw TLC data...
Target per dataset: 1.5 GB (combined target: 6.0 GB)

Sampling yellow...
  yellow: 78,463,852 rows written, 1.50 GB -> data/processing/yellow_processing.parquet
Sampling green...
  green: 1,067,809 rows written, 0.03 GB -> data/processing/green_processing.parquet
Sampling fhv...
  fhv: 39,239,405 rows written, 0.48 GB -> data/processing/fhv_processing.parquet
Sampling fhvhv...
  fhvhv: 62,122,411 rows written, 1.51 GB -> data/processing/fhvhv_processing.parquet

PROCESSING SET SUMMARY
  yellow: 1.50 GB
  green: 0.03 GB
  fhv: 0.48 GB
  fhvhv: 1.51 GB

Total processing set size: 3.52 GB
Target met (>= 3GB).

Suggested report sentence:
"The processing dataset (3.52 GB) was constructed by randomly sampling row groups (seed=42) from each of the four raw TLC datasets (Yellow, Green, FHV, FHV-HV), using the same representative sampling methodology applied during Part 1 exploratory analysis, to ensure methodological consistency between t